In [2]:
import pandas as pd

df = pd.read_csv('../data/raw/stroke.csv')

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
df.head()

Shape: (5110, 12)

Data types:
id                     int64
gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
print("Stroke counts:")
print(df['stroke'].value_counts())
print("\nStroke proportions:")
print(df['stroke'].value_counts(normalize=True))

Stroke counts:
stroke
0    4861
1     249
Name: count, dtype: int64

Stroke proportions:
stroke
0    0.951272
1    0.048728
Name: proportion, dtype: float64


In [4]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


In [5]:
df = df.drop(columns=['id'])

print("Gender counts before drop:", df['gender'].value_counts().to_dict())
df = df[df['gender'] != 'Other'].reset_index(drop=True)
print("Gender counts after drop:", df['gender'].value_counts().to_dict())

Gender counts before drop: {'Female': 2994, 'Male': 2115, 'Other': 1}
Gender counts after drop: {'Female': 2994, 'Male': 2115}


In [6]:
df['bmi'] = df['bmi'].fillna(df['bmi'].median())
print("Remaining nulls:", df.isnull().sum().sum())

Remaining nulls: 0


In [7]:
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
for col in categorical_cols:
    print(col, '->', df[col].unique())

gender -> ['Male' 'Female']
ever_married -> ['Yes' 'No']
work_type -> ['Private' 'Self-employed' 'Govt_job' 'children' 'Never_worked']
Residence_type -> ['Urban' 'Rural']
smoking_status -> ['formerly smoked' 'never smoked' 'smokes' 'Unknown']


In [8]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

Shape after encoding: (5109, 16)


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,228.69,36.6,1,True,True,False,True,False,False,True,True,False,False
1,61.0,0,0,202.21,28.1,1,False,True,False,False,True,False,False,False,True,False
2,80.0,0,1,105.92,32.5,1,True,True,False,True,False,False,False,False,True,False
3,49.0,0,0,171.23,34.4,1,False,True,False,True,False,False,True,False,False,True
4,79.0,1,0,174.12,24.0,1,False,True,False,False,True,False,False,False,True,False


In [12]:
import sys
!{sys.executable} -m pip install scikit-learn xgboost imbalanced-learn shap joblib


   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -----------


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_encoded.drop(columns=['stroke'])
y = df_encoded['stroke']
feature_order = list(X.columns)
print("Features:", feature_order)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print("Train stroke cases:", y_train.sum(), "/ Test stroke cases:", y_test.sum())

Features: ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'gender_Male', 'ever_married_Yes', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'Residence_type_Urban', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']
Train: (4087, 15), Test: (1022, 15)
Train stroke cases: 199 / Test stroke cases: 50


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, recall_score, roc_auc_score

baseline = RandomForestClassifier(n_estimators=100, random_state=42)
baseline.fit(X_train_scaled, y_train)
baseline_preds = baseline.predict(X_test_scaled)

print("--- Baseline (no imbalance handling) ---")
print(classification_report(y_test, baseline_preds))
print("Recall on stroke cases:", recall_score(y_test, baseline_preds))


--- Baseline (no imbalance handling) ---
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   macro avg       0.48      0.50      0.49      1022
weighted avg       0.90      0.95      0.93      1022

Recall on stroke cases: 0.0


In [13]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_resampled.value_counts().to_dict())


Before SMOTE: {0: 3888, 1: 199}
After SMOTE: {0: 3888, 1: 3888}


In [14]:
from xgboost import XGBClassifier

rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_resampled, y_train_resampled)
rf_preds = rf_smote.predict(X_test_scaled)

xgb_smote = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_smote.fit(X_train_resampled, y_train_resampled)
xgb_preds = xgb_smote.predict(X_test_scaled)

print("--- Random Forest + SMOTE ---")
print(classification_report(y_test, rf_preds))
print("Recall:", recall_score(y_test, rf_preds), "| ROC-AUC:", roc_auc_score(y_test, rf_smote.predict_proba(X_test_scaled)[:, 1]))

print("\n--- XGBoost + SMOTE ---")
print(classification_report(y_test, xgb_preds))
print("Recall:", recall_score(y_test, xgb_preds), "| ROC-AUC:", roc_auc_score(y_test, xgb_smote.predict_proba(X_test_scaled)[:, 1]))

--- Random Forest + SMOTE ---
              precision    recall  f1-score   support

           0       0.96      0.96      0.96       972
           1       0.16      0.16      0.16        50

    accuracy                           0.92      1022
   macro avg       0.56      0.56      0.56      1022
weighted avg       0.92      0.92      0.92      1022

Recall: 0.16 | ROC-AUC: 0.7509156378600823

--- XGBoost + SMOTE ---
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       972
           1       0.19      0.14      0.16        50

    accuracy                           0.93      1022
   macro avg       0.57      0.55      0.56      1022
weighted avg       0.92      0.93      0.92      1022

Recall: 0.14 | ROC-AUC: 0.7717901234567901


In [15]:
rf_weighted = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_weighted.fit(X_train_scaled, y_train)  
weighted_preds = rf_weighted.predict(X_test_scaled)

print("--- Random Forest + class_weight='balanced' (no SMOTE) ---")
print(classification_report(y_test, weighted_preds))
print("Recall:", recall_score(y_test, weighted_preds), "| ROC-AUC:", roc_auc_score(y_test, rf_weighted.predict_proba(X_test_scaled)[:, 1]))


--- Random Forest + class_weight='balanced' (no SMOTE) ---
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       972
           1       0.23      0.10      0.14        50

    accuracy                           0.94      1022
   macro avg       0.59      0.54      0.55      1022
weighted avg       0.92      0.94      0.93      1022

Recall: 0.1 | ROC-AUC: 0.7987448559670782


In [16]:
import os, joblib

best_model = rf_smote

output_dir = '../app/modules/stroke'
os.makedirs(output_dir, exist_ok=True)

joblib.dump(best_model, os.path.join(output_dir, 'model.pkl'))
joblib.dump(scaler, os.path.join(output_dir, 'scaler.pkl'))
with open(os.path.join(output_dir, 'feature_order.json'), 'w') as f:
    json.dump(feature_order, f)

print("Saved model.pkl, scaler.pkl, feature_order.json to app/modules/stroke/")

Saved model.pkl, scaler.pkl, feature_order.json to app/modules/stroke/
